# AMRG baseline: re-implementation on our split (Colab GPU)

Reviewers asked that the comparison model run on **our** dataset. This notebook
re-implements **AMRG** (Sung et al., *AMRG: Extend Vision Language Models for
Automatic Mammography Report Generation*, [arXiv:2508.09225](https://arxiv.org/abs/2508.09225)).
The authors released no code, weights or prompt, so everything below is rebuilt
from the paper and then trained and tested on exactly the same
train / validation / test split as `medgemma-train-eval.ipynb`.

**Rule used:** where the paper states a setting, we use the paper's value. Where the
paper is silent, we use the same value as our own training notebook, so the only
differences between the two runs are the ones AMRG actually specifies.

| Setting | AMRG (paper) | Used here | Source |
|---|---|---|---|
| Base model | MedGemma-4B-it | `google/medgemma-4b-it` | paper §3.2 |
| Image preprocessing | Otsu crop → 512×512 → flip left breasts → CLAHE (LAB, 8×8, clip 2.0) | same | paper §3.1 |
| Augmentation | none mentioned | none | paper |
| LoRA | r=32, α=16, dropout 0.05, all linear layers, `embed_tokens` + `lm_head` trainable | same | paper §3.2, Table 1 |
| Optimiser, LR, epochs | AdamW, 1e-4, 20 epochs | same | paper §4.1 |
| Batch | 4 × grad. accumulation 8 | same | paper §4.1 |
| Decoding | temperature 0.1 | temperature 0.1 (pure temperature sampling) | paper, Table 1 caption |
| Prompt / report format | not published | our prompt (`prompts-cot-zeroshot.yaml`, `zero_shot`) | ours |
| Quantisation | not stated | 4-bit NF4 QLoRA, bf16 compute | ours |
| LR schedule, warm-up, grad-norm clip | not stated | linear, 3 % warm-up, 0.3 | ours |
| Checkpoint selection | not stated | best `eval_loss` on validation, early stopping (patience 5) | ours |
| Max new tokens | not stated | 1024 | ours |
| Seed | not stated | 2 | ours |

**One deliberate change from our notebook:** validation runs once per **epoch**
instead of every 50 steps. With AMRG's effective batch of 32 there are ~115 steps per
epoch, so every-50-steps would mean ~46 passes over the 1,000 validation images, and
a patience of 5 evaluations would stop training after ~2 epochs, well short of AMRG's 20.

**Runtime:** A100 GPU, High-RAM. 20 epochs over 3,670 training records is ~2,300
optimiser steps; expect several hours. Checkpoints are written to Drive, so a
disconnected run resumes where it stopped when you run the notebook again.

**Outputs** (in `results/amrg/`): `eval_output.txt`, `eval_predictions.jsonl`
(same schema as ours, so `aware-triage-method.ipynb` can read it),
`eval_metrics.json`, `comparison.csv`, `paired_difference.csv`.

> [!NOTE]
> `google/medgemma-4b-it` is a different HuggingFace repo from the
> `medgemma-1.5-4b-it` we use. Accept its terms on the model page with the same
> account as your `HF_TOKEN` before running.

In [ ]:
!pip install -q -U bitsandbytes evaluate rouge-score trl peft pycocoevalcap

In [ ]:
import json
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from datasets import load_from_disk
from datasets.formatting.formatting import LazyBatch
from huggingface_hub import login
from peft import LoraConfig, PeftModel
from PIL import Image
from pycocoevalcap.cider.cider import Cider
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from tqdm import tqdm
from transformers import (
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    Gemma3ForConditionalGeneration,
    Gemma3Processor,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

SEED = 2
set_seed(SEED)

## Configuration

Values from the AMRG paper are grouped first; values the paper does not give
come from `medgemma-train-eval.ipynb`.

In [ ]:
# ---- From the AMRG paper (arXiv:2508.09225) --------------------------------
MODEL_ID = "google/medgemma-4b-it"         # §3.2: MedGemma-4B-it
IMAGE_SIZE = 512                            # §3.1: resize to 512 x 512
CLAHE_CLIP_LIMIT = 2.0                      # §3.1
CLAHE_TILE_GRID = (8, 8)                    # §3.1
LORA_R = 32                                 # §3.2 / Table 1 best config
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
NUM_EPOCHS = 20                             # §4.1
TRAIN_BATCH_SIZE = 4                        # §4.1
GRAD_ACCUM_STEPS = 8                        # §4.1
LEARNING_RATE = 1e-4                        # §4.1 (AdamW)
TEMPERATURE = 0.1                           # Table 1 caption

# ---- Not in the paper: same as medgemma-train-eval.ipynb --------------------
QUANTIZE_4BIT = True
LR_SCHEDULER = "linear"
WARMUP = 0.03
MAX_GRAD_NORM = 0.3
EVAL_BATCH_SIZE = 8
EARLY_STOPPING_PATIENCE = 5
MAX_NEW_TOKENS = 1024
EVAL_STRATEGY = "epoch"   # ours is every 50 steps; see the note at the top

# ---- Run control -------------------------------------------------------------
TRAIN = True          # False: skip training and load the saved best adapter from Drive
PUSH_TO_HUB = True    # pushes a PRIVATE repo; set False to keep it local/Drive only
HUB_MODEL_ID = "medgemma-4b-it-amrg-lora-dmid-vindr"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
ROOT_DIR = Path("/content/drive/MyDrive/MedGemma2026/main")
DATA_DIR = ROOT_DIR / "data"
CONFIG_DIR = ROOT_DIR / "config"
SPLIT_DIR = DATA_DIR / "split"
VINDR_ANN = DATA_DIR / "vindr-mammo" / "breast-level_annotations.csv"
AMRG_IMG_DIR = DATA_DIR / "amrg-images"             # preprocessed images (cached)
CKPT_DIR = ROOT_DIR / "checkpoints" / HUB_MODEL_ID  # Trainer checkpoints (resumable)
BEST_ADAPTER_DIR = CKPT_DIR / "best-adapter"
OUR_RESULTS_DIR = ROOT_DIR / "results"              # medgemma-train-eval.ipynb outputs
RESULTS_DIR = ROOT_DIR / "results" / "amrg"
LEAKAGE_FILE = ROOT_DIR / "results" / "leakage_check.json"  # optional, from split-leakage-check.ipynb

for d in (AMRG_IMG_DIR, CKPT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
from google.colab import userdata

login(userdata.get("HF_TOKEN"))

## Data

Same `DatasetDict` as `medgemma-train-eval.ipynb` (written by `medgemma-split.ipynb`).
Oversampled duplicates in **train** are kept as plain repeats: AMRG uses no
augmentation, so the `augment` flag is ignored here.

In [ ]:
data = load_from_disk(str(SPLIT_DIR))
print({k: len(v) for k, v in data.items()})
data

## AMRG image preprocessing (paper §3.1)

Applied to the same PNGs our model uses:

1. **Otsu threshold** on the grayscale image, **crop** to the bounding box of the
   breast, **resize to 512×512** (the paper resizes to a fixed square, so the aspect
   ratio is not kept). We crop to the largest connected component of the Otsu mask,
   so small burned-in labels do not widen the box.
2. **Flip** so every breast faces the same way. The paper flips by the laterality
   label. We flip from the image itself (tissue mass on the right half ⇒ chest wall
   on the right ⇒ flip), which gives the same result on standard-orientation
   images and also works for DMID, whose laterality is not in our files. The
   check below compares the flips with VinDr's laterality labels.
3. **CLAHE** on the L channel in LAB space, tile grid 8×8, clip limit 2.0.

Processed images are written once to `data/amrg-images/` and kept in RAM
(~4.5 GB for ~5,700 images) so training does not read from Drive every epoch.

In [ ]:
_CLAHE = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID)


def amrg_preprocess(path: str) -> tuple[np.ndarray, bool]:
    """Return (RGB uint8 IMAGE_SIZE x IMAGE_SIZE image, flipped?)."""
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(path)

    # 1) Otsu mask -> crop to the largest component's bounding box -> square resize
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n > 1:
        k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        x, y, w, h = stats[k, :4]
        gray = gray[y : y + h, x : x + w]
    gray = cv2.resize(gray, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

    # 2) Orientation: put the chest wall on the left so all breasts face right
    half = IMAGE_SIZE // 2
    flipped = bool(gray[:, half:].astype(np.int64).sum() > gray[:, :half].astype(np.int64).sum())
    if flipped:
        gray = cv2.flip(gray, 1)

    # 3) CLAHE on L in LAB space
    lab = cv2.cvtColor(cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    rgb = cv2.cvtColor(cv2.merge([_CLAHE.apply(l), a, b]), cv2.COLOR_LAB2RGB)
    return rgb, flipped


def amrg_path(orig: str) -> Path:
    p = Path(orig)
    return AMRG_IMG_DIR / p.parent.name / p.name   # keeps VinDr <study>/<image> unique

In [ ]:
MANIFEST = AMRG_IMG_DIR / "manifest.csv"
manifest = (
    pd.read_csv(MANIFEST).set_index("orig")["flipped"].to_dict() if MANIFEST.exists() else {}
)

all_paths = sorted({p for name in data for p in data[name]["image"]})
IMAGE_CACHE: dict[str, np.ndarray] = {}
for orig in tqdm(all_paths, desc="AMRG preprocessing"):
    out = amrg_path(orig)
    if out.exists() and orig in manifest:
        IMAGE_CACHE[orig] = cv2.cvtColor(cv2.imread(str(out)), cv2.COLOR_BGR2RGB)
        continue
    rgb, flipped = amrg_preprocess(orig)
    out.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    manifest[orig] = flipped
    IMAGE_CACHE[orig] = rgb

pd.DataFrame({"orig": list(manifest), "flipped": list(manifest.values())}).to_csv(MANIFEST, index=False)
print(f"{len(IMAGE_CACHE)} images ready, {sum(manifest[p] for p in all_paths)} flipped")

In [ ]:
# Sanity check 1: flips vs. VinDr laterality labels. Agreement near 100 % (or near
# 0 %, if the PNGs were already mirrored) means the orientation is consistent;
# near 50 % would mean the image heuristic is not working.
lat_of = pd.read_csv(VINDR_ANN, dtype={"image_id": str}).set_index("image_id")["laterality"].to_dict()
vindr_paths = [p for p in all_paths if "/vindr-mammo/" in p]
agree = [manifest[p] == (lat_of.get(Path(p).stem) == "L") for p in vindr_paths]
print(f"VinDr: flip agrees with laterality label on {np.mean(agree):.1%} of {len(agree)} images")
for src in ("dmid", "vindr-mammo"):
    ps = [p for p in all_paths if f"/{src}/" in p]
    print(f"  {src:<12} flipped {np.mean([manifest[p] for p in ps]):.1%} of {len(ps)}")

# Sanity check 2: before / after for a few images
rng_show = np.random.default_rng(SEED)
show = [all_paths[i] for i in rng_show.choice(len(all_paths), 4, replace=False)]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j, p in enumerate(show):
    axes[0, j].imshow(cv2.imread(p, cv2.IMREAD_GRAYSCALE), cmap="gray")
    axes[0, j].set_title(Path(p).stem[:14], fontsize=8)
    axes[1, j].imshow(IMAGE_CACHE[p])
    axes[1, j].set_title(f"AMRG (flipped={manifest[p]})", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## Prompt and transforms

Same prompt and report format as our model (AMRG did not publish theirs), so the
BI-RADS and density parsers read both models' outputs the same way.

In [ ]:
with open(CONFIG_DIR / "prompts-cot-zeroshot.yaml") as stream:
    PROMPT = yaml.safe_load(stream)["zero_shot"]


def build_messages(report: str | None = None) -> list[dict]:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": PROMPT["system"]}]},
        {
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": PROMPT["user_instruction"]}],
        },
    ]
    if report is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": report}]})
    return messages


def transform_with_report(batch: LazyBatch) -> LazyBatch:
    # train and validation (validation loss needs the report); no augmentation
    batch["image"] = [Image.fromarray(IMAGE_CACHE[p]) for p in batch["image"]]
    batch["messages"] = [build_messages(t) for t in batch["text"]]
    return batch


def transform_prompt_only(batch: LazyBatch) -> LazyBatch:
    # test: generation from the prompt only
    batch["image"] = [Image.fromarray(IMAGE_CACHE[p]) for p in batch["image"]]
    batch["messages"] = [build_messages() for _ in batch["text"]]
    return batch


data["train"].set_transform(transform_with_report)
data["validation"].set_transform(transform_with_report)
data["test"].set_transform(transform_prompt_only)


def extract_birads_level(text: str) -> str:
    match = re.search(r"BI[-\s]?RADS:\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
    return (
        re.sub(r"\s*,\s*", " and ", match.group(1).strip().lower())
        if match
        else "unknown"
    )


def extract_acr_density(text: str) -> str | None:
    match = re.search(r"\(ACR[\s-]+([A-D])\s*\)", text, re.IGNORECASE)
    return match.group(1).upper() if match else None

## Model and LoRA (paper §3.2)

`target_modules="all-linear"` puts LoRA on every linear layer of both the SigLIP
vision encoder and the language decoder (attention q/k/v/o and MLP gate/up/down),
which is what the paper describes. `embed_tokens` and `lm_head` are trained in full,
as in the paper and in our notebook.

In [ ]:
model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
if QUANTIZE_4BIT:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_storage=torch.bfloat16,
    )

model = Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID, **model_kwargs)
processor = Gemma3Processor.from_pretrained(MODEL_ID, use_fast=True)
processor.tokenizer.padding_side = "right"

peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules="all-linear",
    modules_to_save=["lm_head", "embed_tokens"],
    ensure_weight_tying=True,
)

In [ ]:
def collate_fn(examples: list[dict]):
    texts, images = [], []
    for example in examples:
        images.append([example["image"]])
        texts.append(
            processor.apply_chat_template(
                example["messages"], add_generation_prompt=False, tokenize=False
            ).strip()
        )

    batch = processor(images=images, text=texts, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    image_token_id = processor.tokenizer.convert_tokens_to_ids(
        processor.tokenizer.special_tokens_map["boi_token"]
    )
    labels[
        (labels == processor.tokenizer.pad_token_id)
        | (labels == image_token_id)
        | (labels == 262144)
    ] = -100
    batch["labels"] = labels
    return batch

## Training

Checkpoints go to Drive (`checkpoints/medgemma-4b-it-amrg-lora-dmid-vindr/`) and
training resumes from the latest one if it exists. Each checkpoint holds the LoRA
weights plus the full `embed_tokens` / `lm_head` and optimiser state (several GB),
so only the best and latest are kept (`save_total_limit=2`). Make sure Drive has
~20 GB free.

In [ ]:
if TRAIN:
    args = SFTConfig(
        output_dir=str(CKPT_DIR),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        optim="adamw_torch",
        lr_scheduler_type=LR_SCHEDULER,
        warmup_steps=WARMUP,
        max_grad_norm=MAX_GRAD_NORM,
        bf16=True,
        eval_strategy=EVAL_STRATEGY,
        save_strategy=EVAL_STRATEGY,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_steps=10,
        seed=SEED,
        remove_unused_columns=False,
        label_names=["labels"],
        report_to="none",
        push_to_hub=PUSH_TO_HUB,
        hub_strategy="end",
        hub_model_id=HUB_MODEL_ID,
        hub_private_repo=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataset_kwargs={"skip_prepare_dataset": True},
    )

    trainer = SFTTrainer(
        model,
        args=args,
        data_collator=collate_fn,
        train_dataset=data["train"],
        eval_dataset=data["validation"],
        processing_class=processor,
        peft_config=peft_config,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    resume = any(CKPT_DIR.glob("checkpoint-*"))
    print(f"Resuming from the latest checkpoint in {CKPT_DIR}" if resume else "Starting fresh")
    trainer.train(resume_from_checkpoint=True if resume else None)

    # best checkpoint is loaded at the end; keep a copy that evaluation can reload
    trainer.save_model(str(BEST_ADAPTER_DIR))
    print(f"Best adapter saved to {BEST_ADAPTER_DIR}")

In [ ]:
if TRAIN:
    hist = trainer.state.log_history
    tr = [(h["epoch"], h["loss"]) for h in hist if "loss" in h and "eval_loss" not in h]
    ev = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]

    fig, ax = plt.subplots(figsize=(12, 4))
    if tr:
        ax.plot(*zip(*tr), lw=1.2, label="train loss")
    if ev:
        ax.plot(*zip(*ev), marker="o", lw=1.5, label="validation loss")
        best_epoch, best_loss = min(ev, key=lambda x: x[1])
        ax.axvline(best_epoch, color="red", ls="--", alpha=0.7,
                   label=f"best epoch {best_epoch:.0f} (eval_loss={best_loss:.4f})")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title("AMRG re-implementation: train vs validation loss")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "loss_curve.png", dpi=150, bbox_inches="tight")
    plt.show()

## Generate reports on the test split

Decoding follows the paper: sampling at temperature 0.1, with top-k and top-p
switched off so temperature is the only change from the raw distribution.

The closed-set BI-RADS / ACR probabilities are computed the same way as in our
notebook, from the model's **raw** logits (`output_logits=True`), so they are not
affected by the temperature.

In [ ]:
if TRAIN:
    model = trainer.model
else:
    model = PeftModel.from_pretrained(model, str(BEST_ADAPTER_DIR))
    print(f"Loaded adapter from {BEST_ADAPTER_DIR}")
model.eval()
tokenizer = processor.tokenizer

In [ ]:
ACR_CHOICES = ["A", "B", "C", "D"]
BIRADS_CHOICES = ["1", "2", "3", "4", "5"]


def _cumulative_spans(token_ids):
    spans, prev = [], ""
    for k in range(len(token_ids)):
        cur = tokenizer.decode(token_ids[: k + 1], skip_special_tokens=True)
        spans.append((len(prev), len(cur)))
        prev = cur
    return spans, prev


def _step_for_char(spans, char_idx):
    for t, (s, e) in enumerate(spans):
        if s <= char_idx < e:
            return t
    return None


def _candidate_ids(choices, chosen_char, chosen_id):
    for prefix in (" ", ""):
        ids = {c: tokenizer.encode(prefix + c, add_special_tokens=False) for c in choices}
        if all(len(v) == 1 for v in ids.values()) and ids.get(chosen_char, [None])[0] == chosen_id:
            return {c: v[0] for c, v in ids.items()}, True
    ids = {c: tokenizer.encode(c, add_special_tokens=False) for c in choices}
    return {c: (v[0] if len(v) == 1 else None) for c, v in ids.items()}, False


def _closed_set_prob(logits_per_step, char_idx, spans, gen_ids, choices):
    t = _step_for_char(spans, char_idx)
    if t is None or t >= len(logits_per_step):
        return None
    chosen_id = int(gen_ids[t])
    chosen_char = tokenizer.decode([chosen_id]).strip()
    cand, form_ok = _candidate_ids(choices, chosen_char, chosen_id)
    ids = [cand[c] for c in choices]
    if any(j is None for j in ids):
        return None
    logits = logits_per_step[t][ids].float()
    probs = torch.softmax(logits, dim=-1)
    pred = choices[int(torch.argmax(probs))]
    return {
        "pred": pred,
        "conf": float(probs[choices.index(pred)]),
        "dist": dict(zip(choices, [float(p) for p in probs])),
        "logits": [float(x) for x in logits],
        "form_ok": form_ok,
    }


def score_report(generated_ids, logits_per_step):
    spans, text = _cumulative_spans(generated_ids.tolist())
    out = {"text": text, "acr": None, "birads": None}
    m = re.search(r"\(ACR[\s-]+([A-D])\s*\)", text, re.IGNORECASE)
    if m:
        out["acr"] = _closed_set_prob(logits_per_step, m.start(1), spans, generated_ids, ACR_CHOICES)
    m = re.search(r"BI[-\s]?RADS:\s*([1-5])", text, re.IGNORECASE)
    if m:
        out["birads"] = _closed_set_prob(logits_per_step, m.start(1), spans, generated_ids, BIRADS_CHOICES)
    return out

In [ ]:
def run_eval(split, desc):
    rows = []
    image_paths = split.data.column("image").to_pylist()   # raw paths (bypass the transform)
    sources = split.data.column("source").to_pylist()
    torch.manual_seed(SEED)
    for i in tqdm(range(len(split)), desc=desc):
        sample = split[i]
        inputs = processor(
            images=[[sample["image"]]],
            text=[processor.apply_chat_template(
                sample["messages"], add_generation_prompt=True, tokenize=False
            ).strip()],
            return_tensors="pt",
            padding=True,
        ).to(model.device)
        in_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_k=0,
                top_p=1.0,
                return_dict_in_generate=True,
                output_logits=True,
            )

        gen_ids = out.sequences[0, in_len:].detach().cpu()
        logits = [s[0].detach().float().cpu() for s in out.logits]
        del out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        sc = score_report(gen_ids, logits)
        rows.append({
            "i": i,
            "image_id": Path(image_paths[i]).stem,
            "image_path": image_paths[i],
            "source": sources[i],
            "text": sc["text"],
            "ref": sample["text"],
            "acr": sc["acr"],
            "birads": sc["birads"],
        })
    return rows


rows = run_eval(data["test"], "AMRG: generating + scoring (test)")

with open(RESULTS_DIR / "eval_predictions.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

with open(RESULTS_DIR / "eval_output.txt", "w") as f:
    for r in rows:
        f.write(f"\n--- Sample {r['i']}  [{r['image_id']}]  {r['image_path']} ---\n")
        f.write(f"Original report: {r['ref']}\n")
        f.write(f"Generated report: {r['text']}\n")
        if r["birads"]:
            f.write(f"BI-RADS pred={r['birads']['pred']} conf={r['birads']['conf']:.3f} dist={r['birads']['dist']}\n")
        if r["acr"]:
            f.write(f"ACR pred={r['acr']['pred']} conf={r['acr']['conf']:.3f} dist={r['acr']['dist']}\n")

print(f"Saved {len(rows)} predictions to {RESULTS_DIR}")

## Metrics: AMRG vs. our model on the same test images

One metric function is applied to both prediction files, so the numbers are
computed identically. It covers AMRG's metric set (BLEU-1, ROUGE-1/2/L, METEOR,
CIDEr, word-level F1, BI-RADS accuracy, density accuracy) plus the imbalance-aware
metrics from our notebook (macro-F1, balanced accuracy), with bootstrap 95 % CIs.

Each model is scored on these subsets of the test split:

- **all**: the full test split,
- **dmid**: DMID images only (closest to AMRG's own setting),
- **clean**: only if `results/leakage_check.json` exists (written by
  `split-leakage-check.ipynb`): test images with no detected same-patient leakage.
  Without that file this subset is skipped; nothing else changes.

CIDEr uses `pycocoevalcap` on lower-cased word tokens (no Java PTB tokenizer).
Its IDF statistics come from the reference reports of the subset being scored, so
CIDEr values are only comparable within the same subset.

In [ ]:
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
bleu = evaluate.load("bleu")
_rng = np.random.default_rng(SEED)


def _tok(s: str) -> str:
    return " ".join(re.findall(r"\w+", s.lower()))


def word_level_f1(prediction: str, reference: str) -> float:
    p, r = prediction.lower().split(), reference.lower().split()
    common = sum((Counter(p) & Counter(r)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(p), common / len(r)
    return 2 * precision * recall / (precision + recall)


def boot(y_true, y_pred, metric, n=1000):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    point = float(metric(y_true, y_pred))
    vals = []
    for _ in range(n):
        ii = _rng.integers(0, len(y_true), size=len(y_true))
        vals.append(metric(y_true[ii], y_pred[ii]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return point, float(lo), float(hi)


def macro_f1(yt, yp):
    return f1_score(yt, yp, average="macro", zero_division=0)


def compute_all_metrics(preds: list[str], refs: list[str]) -> dict:
    pairs = [(p, r) for p, r in zip(preds, refs) if p.strip()]
    vp, vr = (list(x) for x in zip(*pairs)) if pairs else ([], [])
    out = {"n": len(preds), "n_nonempty": len(vp)}
    if vp:
        rg = rouge.compute(predictions=vp, references=vr)
        out.update({"rouge1": rg["rouge1"], "rouge2": rg["rouge2"], "rougeL": rg["rougeL"]})
        out["meteor"] = meteor.compute(predictions=vp, references=vr)["meteor"]
        for order in (1, 4):
            try:
                out[f"bleu{order}"] = bleu.compute(predictions=vp, references=vr, max_order=order)["bleu"]
            except ZeroDivisionError:
                out[f"bleu{order}"] = 0.0
        out["cider"] = float(Cider().compute_score(
            {i: [_tok(r)] for i, r in enumerate(vr)},
            {i: [_tok(p)] for i, p in enumerate(vp)},
        )[0])
        out["word_f1"] = float(np.mean([word_level_f1(p, r) for p, r in zip(vp, vr)]))

    tb = [extract_birads_level(r) for r in refs]
    pb = [extract_birads_level(p) for p in preds]
    ta = [extract_acr_density(r) or "unknown" for r in refs]
    pa = [extract_acr_density(p) or "unknown" for p in preds]
    for name, yt, yp in (("birads", tb, pb), ("density", ta, pa)):
        out[f"{name}_acc"] = boot(yt, yp, accuracy_score)
        out[f"{name}_macro_f1"] = boot(yt, yp, macro_f1)
        out[f"{name}_balanced_acc"] = boot(yt, yp, balanced_accuracy_score)
    return out


def load_rows(path: Path) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]


CLEAN_TEST_IMAGES = None
if LEAKAGE_FILE.exists():
    with open(LEAKAGE_FILE) as f:
        CLEAN_TEST_IMAGES = set(json.load(f)["clean_test_images"])
    print(f"Leakage file found: 'clean' subset has {len(CLEAN_TEST_IMAGES)} test images")
else:
    print("No leakage_check.json: scoring 'all' and 'dmid' subsets only")


def subsets(rows: list[dict]) -> dict[str, list[dict]]:
    out = {
        "all": rows,
        "dmid": [r for r in rows if "/dmid/" in r["image_path"]],
    }
    if CLEAN_TEST_IMAGES is not None:
        out["clean"] = [r for r in rows if r["image_path"] in CLEAN_TEST_IMAGES]
    return out

In [ ]:
models = {"AMRG (re-impl.)": load_rows(RESULTS_DIR / "eval_predictions.jsonl")}
ours_path = OUR_RESULTS_DIR / "eval_predictions.jsonl"
if ours_path.exists():
    models["Ours (DB-ATRG)"] = load_rows(ours_path)
else:
    print(f"Our predictions not found at {ours_path}; run medgemma-train-eval.ipynb first. "
          "Scoring AMRG only.")

# both files must cover the same test images
if len(models) == 2:
    a, b = ({r["image_path"] for r in rs} for rs in models.values())
    assert a == b, f"test sets differ: {len(a ^ b)} images not shared"

all_metrics = {}
table = []
for model_name, rows_m in models.items():
    for subset_name, rs in subsets(rows_m).items():
        m = compute_all_metrics([r["text"] for r in rs], [r["ref"] for r in rs])
        all_metrics[f"{model_name} | {subset_name}"] = m
        row = {"model": model_name, "subset": subset_name, "n": m["n"]}
        for k in ("bleu1", "bleu4", "rouge1", "rouge2", "rougeL", "meteor", "cider", "word_f1"):
            row[k] = round(m.get(k, float("nan")), 4)
        for k in ("birads_acc", "birads_macro_f1", "birads_balanced_acc",
                  "density_acc", "density_macro_f1", "density_balanced_acc"):
            p, lo, hi = m[k]
            row[k] = f"{p:.4f} [{lo:.3f}, {hi:.3f}]"
        table.append(row)

comparison = pd.DataFrame(table).sort_values(["subset", "model"]).reset_index(drop=True)
comparison.to_csv(RESULTS_DIR / "comparison.csv", index=False)
with open(RESULTS_DIR / "eval_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
comparison

### Paired difference (ours − AMRG)

Both models were run on the same images, so a paired bootstrap over images gives
a 95 % CI for the difference. A CI that excludes 0 means the gap is unlikely to be
resampling noise.

In [ ]:
if len(models) == 2:
    amrg_by = {r["image_path"]: r for r in models["AMRG (re-impl.)"]}
    ours_by = {r["image_path"]: r for r in models["Ours (DB-ATRG)"]}

    def per_image(rows_by, keys):
        texts = [rows_by[k]["text"] for k in keys]
        refs = [rows_by[k]["ref"] for k in keys]
        rl = rouge.compute(predictions=[t if t.strip() else " " for t in texts],
                           references=refs, use_aggregator=False)["rougeL"]
        bir = [extract_birads_level(t) == extract_birads_level(r) for t, r in zip(texts, refs)]
        acr = [extract_acr_density(t) == extract_acr_density(r) for t, r in zip(texts, refs)]
        return {"rougeL": np.array(rl), "birads_acc": np.array(bir, float),
                "density_acc": np.array(acr, float)}

    paired = []
    for subset_name, sub_rows in subsets(models["AMRG (re-impl.)"]).items():
        keys = [r["image_path"] for r in sub_rows]
        a, o = per_image(amrg_by, keys), per_image(ours_by, keys)
        for metric in ("rougeL", "birads_acc", "density_acc"):
            d = o[metric] - a[metric]
            idx = _rng.integers(0, len(d), size=(2000, len(d)))
            lo, hi = np.percentile(d[idx].mean(axis=1), [2.5, 97.5])
            paired.append({"subset": subset_name, "metric": metric, "n": len(d),
                           "ours": o[metric].mean(), "amrg": a[metric].mean(),
                           "diff": d.mean(), "ci_low": lo, "ci_high": hi})
    paired_df = pd.DataFrame(paired).round(4)
    paired_df.to_csv(RESULTS_DIR / "paired_difference.csv", index=False)
    display(paired_df)

## Next

- To run the triage simulation on AMRG's predictions, call
  `run_simulation(jsonl_path=str(RESULTS_DIR / "eval_predictions.jsonl"), output_dir=str(RESULTS_DIR))`
  in `aware-triage-method.ipynb`.
- Report AMRG's numbers from this notebook, not the paper's Table 2: the paper scored
  52 DMID test images with its own report format and parser.